In [47]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from scipy.stats import fisher_exact, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)

figures = Path("../04_FIGURES")
results = Path("../05_RESULTS")
data    = Path("../01_DATA/test_train_split")
selected = Path("../01_DATA/selected")
figures.mkdir(parents=True, exist_ok=True)
results.mkdir(parents=True, exist_ok=True)
data.mkdir(parents=True, exist_ok=True)
selected.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(data / "X_train.csv")
missing_cols = [c for c in train_df.columns if c.endswith("_missing")]

y_train  = train_df["Biopsy"]
x_train  = train_df.drop(columns=["Biopsy"]+ missing_cols)
print(f"Risk Feature Set Shape: {x_train.shape}")

diag_train_df = pd.read_csv(data / "Diag_X_train.csv")
diag_y_train  = diag_train_df["Biopsy"]
diag_x_train  = diag_train_df.drop(columns=["Biopsy"]+ missing_cols)
print(f"Diag Feature Set Shape: {diag_x_train.shape}")
          
cv      = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)          
CORRELATION_THRESHOLD = 0.8

Risk Feature Set Shape: (584, 28)
Diag Feature Set Shape: (584, 31)


In [48]:
def variance_filter(x, threshold=0.01):
    vt = VarianceThreshold(threshold=threshold).fit(x)
    dropped = x.columns[~vt.get_support()]
    print(f"  Variance filter: dropped {len(dropped)} -> {list(dropped)}")
    return x.drop(columns=dropped)

In [49]:
def right_test(col, y, x):
    v = x[col].values
    if set(np.unique(v)) <= {0, 1}:
        table = [[((v==1)&(y==1)).sum(), ((v==1)&(y==0)).sum()],
                 [((v==0)&(y==1)).sum(), ((v==0)&(y==0)).sum()]]
        return fisher_exact(table)[1], "Fisher's exact"
    return mannwhitneyu(v[y==1], v[y==0]).pvalue, "Mann-Whitney U"

In [50]:
def univariate_test(x, y):
    rows = []
    for c in x.columns:
        p, t = right_test(c, y, x)
        rows.append({"Feature": c, "P-value": p, "Test": t})
    table = pd.DataFrame(rows).sort_values("P-value").reset_index(drop=True)
    reject, qvals, _, _ = multipletests(table["P-value"], alpha=0.05, method="fdr_bh")
    table["qvalue"] = qvals
    table["fdr_significant"] = reject
    return table

In [51]:
def mutual_information_test(x, y):
    bi_cat_col = [x[c].nunique() <= 10 for c in x.columns]
    mi = mutual_info_classif(x, y, discrete_features=bi_cat_col, random_state=42)
    return pd.Series(mi, index=x.columns).sort_values(ascending=False)

In [52]:
def l1_selection_frequency(x, y, cv_splitter):
    freq = np.zeros(x.shape[1]); n = 0
    for tr_idx, _ in cv_splitter.split(x, y):
        Xtr = StandardScaler().fit_transform(x.iloc[tr_idx])
        l1 = LogisticRegression(l1_ratio=1.0, solver="saga", C=0.1,
                                class_weight="balanced", max_iter=5000).fit(Xtr, y.iloc[tr_idx])
        freq += (np.abs(l1.coef_[0]) > 1e-8); n += 1
    return pd.Series(freq / n, index=x.columns).sort_values(ascending=False)

In [53]:
def rf_permutation_importance(x, y, cv_splitter, n_repeats=5):
    scores = np.zeros(x.shape[1]); n = 0
    for tr_idx, te_idx in cv_splitter.split(x, y):
        rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    min_samples_leaf=3, random_state=0, n_jobs=-1).fit(x.iloc[tr_idx], y.iloc[tr_idx])
        pi = permutation_importance(rf, x.iloc[te_idx], y.iloc[te_idx], scoring="average_precision",
                                    n_repeats=n_repeats, random_state=0, n_jobs=1)
        scores += pi.importances_mean; n += 1
    return pd.Series(scores / n, index=x.columns).sort_values(ascending=False)

In [54]:
def dedup_by_correlation(x, candidates, mi_scores, threshold=CORRELATION_THRESHOLD):
    if len(candidates) < 2:
        return list(candidates), pd.DataFrame()
    cor = x[candidates].corr()
    pairs = cor.where(np.triu(np.ones(cor.shape), k=1).astype(bool)).stack().reset_index()
    pairs.columns = ["feature_1", "feature_2", "correlation"]
    pairs = pairs[pairs["correlation"].abs() > threshold].sort_values("correlation", key=abs, ascending=False)
    to_drop = set()
    for _, row in pairs.iterrows():
        weaker = row.feature_1 if mi_scores.get(row.feature_1, 0) < mi_scores.get(row.feature_2, 0) else row.feature_2
        to_drop.add(weaker)
    return [f for f in candidates if f not in to_drop], pairs

In [55]:
def vif_table(x, feats):
    if len(feats) < 2:
        return pd.Series(dtype=float)
    check = x[feats].assign(_const=1)
    return pd.Series([variance_inflation_factor(check.values, i) for i in range(check.shape[1] - 1)],
                     index=feats).sort_values(ascending=False)

In [56]:
def validate_selection(x_full, y, final_feats, cv):
    model = LogisticRegression(class_weight="balanced", max_iter=5000)
    score_all      = cross_val_score(model, x_full, y, cv=cv, scoring="average_precision")
    score_selected = cross_val_score(model, x_full[final_feats], y, cv=cv, scoring="average_precision")
    return score_all, score_selected

In [57]:
def run_feature_selection_pipeline(x_raw, y, branch_name, top_n=15, l1_stability_threshold=0.5):
    print(f"\n{'='*90}\n{branch_name}  ({x_raw.shape[1]} candidate features)\n{'='*90}")

    x = variance_filter(x_raw)

    uni_table = univariate_test(x, y)
    fdr_feats = uni_table.loc[uni_table["fdr_significant"], "Feature"].tolist()
    print(f"  {len(fdr_feats)} of {len(uni_table)} features survive FDR correction")

    mi_series = mutual_information_test(x, y)
    top_mi_feats = mi_series.head(top_n).index.tolist()

    l1_freq     = l1_selection_frequency(x, y, cv)
    perm_series = rf_permutation_importance(x, y, cv)
    l1_stable_feats = l1_freq[l1_freq >= l1_stability_threshold].index.tolist()
    top_rf_feats    = perm_series.head(top_n).index.tolist()

    # union of everything flagged by at least one signal (for vote-counting)
    all_candidates = sorted(set(fdr_feats) | set(top_mi_feats) | set(l1_stable_feats) | set(top_rf_feats))
    votes = pd.Series({
        f: sum([f in fdr_feats, f in top_mi_feats, f in l1_stable_feats, f in top_rf_feats])
        for f in all_candidates
    })
    print(f"  Candidates with >=1 vote: {len(all_candidates)}")

    # require agreement across at least 2 independent methods
    candidates = votes[votes >= 2].index.tolist()
    print(f"  Candidates with >=2 votes: {len(candidates)} -> {candidates}")

    final_features, redundant_pairs = dedup_by_correlation(x, candidates, mi_series)
    print(f"  After redundancy dedup (MI tiebreaker): {len(final_features)} -> {final_features}")
    if len(redundant_pairs):
        print(f"  Redundant pairs resolved:\n{redundant_pairs.to_string(index=False)}")

    support = pd.DataFrame({
        "Feature": final_features,
        "in_FDR": [f in fdr_feats for f in final_features],
        "in_top_MI": [f in top_mi_feats for f in final_features],
        "in_L1_stable": [f in l1_stable_feats for f in final_features],
        "in_top_RF": [f in top_rf_feats for f in final_features],
        "L1_selection_freq": [round(l1_freq.get(f, 0), 3) for f in final_features],
        "RF_perm_importance": [round(perm_series.get(f, 0), 4) for f in final_features],
    })
    support["n_votes"] = support[["in_FDR", "in_top_MI", "in_L1_stable", "in_top_RF"]].sum(axis=1)
    support = support.sort_values("n_votes", ascending=False)
    print(f"  Evidence behind each final feature:\n{support.to_string(index=False)}")

    vif = vif_table(x_raw, final_features)
    score_all, score_selected = validate_selection(x_raw, y, final_features, cv)
    print(f"  All features:      PR-AUC = {score_all.mean():.3f} +/- {score_all.std():.3f}")
    print(f"  Selected features: PR-AUC = {score_selected.mean():.3f} +/- {score_selected.std():.3f}")
    print(f"  VIF on final set:\n{vif.to_string()}")

    return {
        "x": x, "uni_table": uni_table, "mi_series": mi_series,
        "l1_freq": l1_freq, "perm_series": perm_series, "support": support,
        "final_features": final_features, "vif": vif,
        "score_all": score_all, "score_selected": score_selected,
    }

In [58]:
risk_result = run_feature_selection_pipeline(x_train, y_train, "Risk factors only")
diag_result = run_feature_selection_pipeline(diag_x_train, diag_y_train, "With diagnostics")

final_features      = risk_result["final_features"]
diag_final_features = diag_result["final_features"]


Risk factors only  (28 candidate features)
  Variance filter: dropped 6 -> ['STDs:vaginal condylomatosis', 'STDs:pelvic inflammatory disease', 'STDs:genital herpes', 'STDs:molluscum contagiosum', 'STDs:Hepatitis B', 'STDs:HPV']
  3 of 22 features survive FDR correction
  Candidates with >=1 vote: 19
  Candidates with >=2 votes: 15 -> ['Dx', 'Dx:CIN', 'Dx:Cancer', 'Dx:HPV', 'First sexual intercourse', 'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'Num of pregnancies', 'Number of sexual partners', 'STDs', 'STDs (number)', 'STDs: Number of diagnosis', 'STDs:HIV', 'STDs:syphilis', 'STDs:vulvo-perineal condylomatosis']
  After redundancy dedup (MI tiebreaker): 11 -> ['Dx', 'Dx:CIN', 'Dx:HPV', 'First sexual intercourse', 'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'Num of pregnancies', 'Number of sexual partners', 'STDs (number)', 'STDs:HIV', 'STDs:syphilis']
  Redundant pairs resolved:
    feature_1                          feature_2  correlation
    Dx:Ca

In [62]:
test_df = pd.read_csv(data / "X_test.csv").drop(columns=missing_cols)
x_train[final_features].assign(Biopsy=y_train).to_csv(selected / "x_train_selected.csv", index=False)
test_df[final_features + ["Biopsy"]].to_csv(selected / "x_test_selected.csv", index=False)

diag_test_df = pd.read_csv(data / "Diag_X_test.csv").drop(columns=missing_cols)
diag_x_train[diag_final_features].assign(Biopsy=diag_y_train).to_csv(selected / "diag_x_train_selected.csv", index=False)
diag_test_df[diag_final_features + ["Biopsy"]].to_csv(selected / "diag_x_test_selected.csv", index=False)

risk_result["uni_table"].to_csv(results / "univariate_test_results.csv", index=False)
diag_result["uni_table"].to_csv(results / "diag_univariate_test_results.csv", index=False)

print("Risk factors only final features:", final_features)
print("With diagnostics final features:", diag_final_features)

Risk factors only final features: ['Dx', 'Dx:CIN', 'Dx:HPV', 'First sexual intercourse', 'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'Num of pregnancies', 'Number of sexual partners', 'STDs (number)', 'STDs:HIV', 'STDs:syphilis']
With diagnostics final features: ['Age', 'Citology', 'Dx', 'Dx:CIN', 'First sexual intercourse', 'Hinselmann', 'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'STDs (number)', 'Schiller']
